# L5B15 Surrogate Model

Builds a CatBoost gradient-boosted surrogate to replicate Pega ADM propensity scores for L5B15 (luggage offer), enabling feature-level interpretation.
Also analyses model version drift within the observation window.

Reads from `data/processed/l5b15_decisions.parquet` — run `02_data_ingestion.ipynb` first.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

print("Imports OK")

In [ ]:
PROCESSED_FILE = Path("../data/processed/l5b15_decisions.parquet")
df_model = pd.read_parquet(PROCESSED_FILE)
print(f"Loaded {len(df_model):,} rows, {df_model.shape[1]} columns")

## 7. Surrogate Model Building

We build a CatBoost gradient-boosted surrogate to predict L5B15 propensity scores from the observable customer input features. The goal is not to outperform the Pega ADM model, but to approximate its scoring behavior closely enough to use the surrogate as an explainability proxy (e.g., with SHAP values).

In [ ]:
### 7.1 Feature Preparation
X = df_model.drop(columns=[c for c in [
    "propensity",
    "pxInteractionID",
    "pxSubjectID",
    "modelExecutionID",
    "pyName",
    "TreatmentID",
    "modelPerformance",
    "modelEvidence",
    "modelTechnique"
] if c in df_model.columns]).copy()

y = df_model["propensity"].astype(float).copy()

# drop very sparse and constant columns
X = X.loc[:, X.isna().mean() < 0.95]
X = X.loc[:, X.nunique(dropna=True) > 1]

# normalize obvious missing markers
X = X.replace({"": np.nan, "NaN": np.nan, "nan": np.nan, "None": np.nan})

# try numeric conversion safely
for col in X.columns:
    try:
        X[col] = pd.to_numeric(X[col])
    except Exception:
        pass

# identify categorical and numeric columns
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

# clean categorical columns
for col in cat_cols:
    X[col] = X[col].astype("string").fillna("__MISSING__")

# clean numeric columns
for col in num_cols:
    X[col] = X[col].astype(float)
    X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    X[col] = X[col].fillna(X[col].median())

print(X.shape)
print(len(num_cols), "numeric,", len(cat_cols), "categorical")

In [ ]:
### 7.2 Model Training

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_features = [X_train.columns.get_loc(c) for c in cat_cols]

model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    verbose=100,
    random_seed=42
)

model.fit(X_train, y_train, cat_features=cat_features)
pred = model.predict(X_test)

print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE:", rmse)

### 7.3 Model Evaluation

The RMSE is relatively large compared to the mean propensity level. This is expected: propensity scores in Pega ADM span a wide range and the surrogate is approximating a black-box Bayesian classifier from observable inputs only. Absolute calibration is therefore less critical than whether the surrogate preserves rank order — which we assess next with the actual vs. predicted plot and Spearman correlation.

In [ ]:
### 7.4 Actual vs Predicted Plot
plt.scatter(y_test, pred, alpha=0.2)
plt.xlabel("Actual propensity")
plt.ylabel("Predicted propensity")
plt.title("CatBoost surrogate: actual vs predicted")
plt.show()

The scatter plot shows good rank fidelity overall: the surrogate correctly separates high-propensity from low-propensity cases. However, calibration is imperfect at the tails — extreme propensity values are compressed toward the center, predicting fewer very high or very low scores than the actual distribution contains. This is typical behavior for gradient-boosted surrogates and does not substantially impair their use as ranking tools.

In [ ]:
### 7.5 Rank Fidelity (Spearman Correlation)
rho, _ = spearmanr(y_test, pred)
print("Spearman rho:", rho)

Spearman rank correlation measures agreement in ordering rather than absolute values. Since propensity scores are used operationally to rank offers, a surrogate that preserves rank order captures the most decision-relevant aspect of the original model’s behavior.

A Spearman ρ around 0.9 is a strong result: the surrogate reliably reproduces which customers are ranked higher or lower, even where it does not match the exact numeric propensity value. This makes it suitable as an explainability proxy.

In [ ]:
### 7.6 Feature Importance from Surrogate
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

fi.head(30)

The table shows which variables CatBoost relied on most to reduce prediction error for propensity. A partition column like `pyChannel` ranking highly is suspicious: if all records in this dataset share the same channel (e.g., always "Email"), it carries no real predictive signal and its high importance reflects a data artifact. The next cell verifies whether these partition columns are effectively constant across the extracted records.

In [ ]:
for c in ["pyChannel", "pyDirection", "pyGroup", "pyIssue", "pyName"]:
    if c in df_model.columns:
        print(c, df_model[c].value_counts(dropna=False).head())

## 8. Model Version Analysis

L5B15 scoring events in the 3-day window are associated with multiple model versions. We inspect which versions were active and whether mean propensity differs across them — a sign of model drift that could contribute to the surrogate’s residual error.

In [ ]:
# Model version distribution across all records
print("Model versions in dataset:")
print(df_model["modelVersion"].value_counts(dropna=False))

In [ ]:
# Filter to a specific partition for cleaner drift analysis
df_email = df_model[
    (df_model["pyChannel"] == "Email") &
    (df_model["pyDirection"] == "Outbound") &
    (df_model["pyGroup"] == "Luggage") &
    (df_model["pyIssue"] == "Sales")
].copy()

print(f"Email/Outbound/Luggage/Sales records: {len(df_email):,}")
print(f"Model versions: {df_email['modelVersion'].nunique()}")

In [ ]:
df_email["propensity"].describe()

In [ ]:
df_email.groupby("modelVersion")["propensity"].mean().sort_values(ascending=False)

Mean propensity differs across model versions, confirming **model drift** within the 3-day observation window. When the underlying Pega ADM model updates between scoring events, identical customer inputs can yield different propensity values — introducing variance that a single surrogate cannot account for. This is the most likely driver of the elevated RMSE observed in Section 7.

**Implication**: For more precise surrogate modelling, records should be stratified by model version, or the analysis restricted to a single stable version. The filtered `df_email` extraction above provides a starting point for this.